# GAT-GRU v05 — selected features final

Chỉ chạy sau khi node/edge feature contract được APPROVED. Kiến trúc GAT-GRU vẫn cố định; Optuna chỉ tune training hyperparameters. Đây là notebook GAT-GRU đầu tiên trong nhánh mới được phép đánh giá test.

In [1]:
import sys
from pathlib import Path
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(
    ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json', require_edge=False)
base_cfg = fa.load_baseline_config(ctx)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'artifacts' / 'gatgru' / 'v05_selected_features_final'
print('node:', selection['node_features'])
print('edge:', selection['edge_features'])
print('device:', DEVICE)

node: ['rps_in', 'rps_in_delta', 'rps_per_replica', 'latency_p50_ms', 'latency_p95_ms', 'cpu_cores', 'mem_mib', 'net_rx_kBps', 'net_tx_kBps', 'replicas']
edge: ['call_rate', 'call_ratio', 'edge_latency_p95_ms']
device: cpu


## 1. Tune training hyperparameters; architecture and features remain fixed

In [2]:
best_cfg, study = fa.run_optuna_fixed_architecture(
    ctx, 'gatgru', selection['node_features'], selection['edge_features'], base_cfg,
    OUT / 'tuning', n_trials=24, seed=42, device=DEVICE,
    score_weights=fa.CPU_DIAGNOSTIC_WEIGHTS, parent_version='v04_edge_controls',
)
print('best score:', study.best_value)
best_cfg

[I 2026-10-18 23:41:52,562] A new study created in RDB with name: gatgru_fb5407310c2dfc0d_fixed_arch
[I 2026-10-18 23:52:22,325] Trial 0 finished with value: 0.10636231260851384 and parameters: {'window': 18, 'dropout': 0.11973169683940732, 'lr': 0.0006427207987637223, 'weight_decay': 2.9375384576328313e-06, 'batch_size': 64}. Best is trial 0 with value: 0.10636231260851384.
[I 2026-10-19 00:11:52,135] Trial 1 finished with value: 0.10749157275866558 and parameters: {'window': 24, 'dropout': 0.16648852816008436, 'lr': 0.0007037018186923737, 'weight_decay': 3.5113563139704077e-06, 'batch_size': 128}. Best is trial 0 with value: 0.10636231260851384.
[I 2026-10-19 00:26:54,638] Trial 2 finished with value: 0.10661964842735211 and parameters: {'window': 24, 'dropout': 0.027898772130408367, 'lr': 0.0008001480489146253, 'weight_decay': 1.2562773503807034e-05, 'batch_size': 64}. Best is trial 0 with value: 0.10636231260851384.
[I 2026-10-19 00:41:09,201] Trial 3 finished with value: 0.1062944

best score: 0.10436421938585208


{'window': 18,
 'dropout': 0.008967376666503827,
 'lr': 0.0005352359452218873,
 'batch_size': 32,
 'gat_hidden': 32,
 'heads': 4,
 'gat_layers': 2,
 'gru_hidden': 128,
 'edge_direction': 'both',
 'use_edge_features': True,
 'weight_decay': 1.8854677893580192e-06}

## 2. Five seeds and one final test evaluation

In [3]:
prediction_path, training_meta = fa.run_final(
    ctx, 'gatgru', selection['node_features'], selection['edge_features'], best_cfg, OUT,
    seeds=(0, 1, 2, 3, 4), device=DEVICE, model_name='gatgru_selected',
    parent_version='v04_edge_controls',
)
print(prediction_path)
training_meta

F:\0. UIT\KLTN\thesis-gnn-rl-autoscaling\math-load-testing\experiments\gatgru\v05_selected_features_final\gatgru_selected_predictions_test.npz


{'model': 'gatgru_selected',
 'model_kind': 'gatgru',
 'config': {'window': 18,
  'dropout': 0.008967376666503827,
  'lr': 0.0005352359452218873,
  'batch_size': 32,
  'gat_hidden': 32,
  'heads': 4,
  'gat_layers': 2,
  'gru_hidden': 128,
  'edge_direction': 'both',
  'use_edge_features': True,
  'weight_decay': 1.8854677893580192e-06},
 'seeds': [0, 1, 2, 3, 4],
 'val_mae_scaled': [0.11316465586423874,
  0.11367759108543396,
  0.11427091062068939,
  0.11282231658697128,
  0.11188845336437225],
 'node_features': ['rps_in',
  'rps_in_delta',
  'rps_per_replica',
  'latency_p50_ms',
  'latency_p95_ms',
  'cpu_cores',
  'mem_mib',
  'net_rx_kBps',
  'net_tx_kBps',
  'replicas'],
 'edge_features': ['call_rate', 'call_ratio', 'edge_latency_p95_ms'],
 'targets': ['rps_in', 'cpu_cores'],
 'services': ['frontend',
  'adservice',
  'cartservice',
  'checkoutservice',
  'currencyservice',
  'emailservice',
  'paymentservice',
  'productcatalogservice',
  'recommendationservice',
  'redis-cart',